In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr


# ------------------------------------------------------------------
# 1. File locations
# ------------------------------------------------------------------

SUPPORT_FILENAME = "1.xlsx"
COOPERATION_FILENAME = "main file.csv"
OUTPUT_DIR = Path("outputs2")


def find_input(filename: str) -> Path:
    """Find a data file in the current directory or its upload subfolder."""
    script_directory = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    candidates = (
        Path.cwd() / filename,
        Path.cwd() / "upload" / filename,
        script_directory / filename,
        script_directory / "upload" / filename,
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = "\n  - ".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n  - {searched}")


# ------------------------------------------------------------------
# 2. Complete country universe
# ------------------------------------------------------------------

# The 193 UN member states in common short-name form. India is removed
# because it is the country whose partnerships and external support are tested.
UN_MEMBER_STATES = (
    "Afghanistan", "Albania", "Algeria", "Andorra", "Angola",
    "Antigua and Barbuda", "Argentina", "Armenia", "Australia", "Austria",
    "Azerbaijan", "Bahamas", "Bahrain", "Bangladesh", "Barbados", "Belarus",
    "Belgium", "Belize", "Benin", "Bhutan", "Bolivia",
    "Bosnia and Herzegovina", "Botswana", "Brazil", "Brunei", "Bulgaria",
    "Burkina Faso", "Burundi", "Cabo Verde", "Cambodia", "Cameroon", "Canada",
    "Central African Republic", "Chad", "Chile", "China", "Colombia", "Comoros",
    "Republic of the Congo", "Costa Rica", "Cote d'Ivoire", "Croatia", "Cuba",
    "Cyprus", "Czechia", "North Korea", "DR Congo", "Denmark", "Djibouti",
    "Dominica", "Dominican Republic", "Ecuador", "Egypt", "El Salvador",
    "Equatorial Guinea", "Eritrea", "Estonia", "Eswatini", "Ethiopia", "Fiji",
    "Finland", "France", "Gabon", "Gambia", "Georgia", "Germany", "Ghana",
    "Greece", "Grenada", "Guatemala", "Guinea", "Guinea-Bissau", "Guyana",
    "Haiti", "Honduras", "Hungary", "Iceland", "India", "Indonesia", "Iran",
    "Iraq", "Ireland", "Israel", "Italy", "Jamaica", "Japan", "Jordan",
    "Kazakhstan", "Kenya", "Kiribati", "Kuwait", "Kyrgyzstan", "Laos", "Latvia",
    "Lebanon", "Lesotho", "Liberia", "Libya", "Liechtenstein", "Lithuania",
    "Luxembourg", "Madagascar", "Malawi", "Malaysia", "Maldives", "Mali",
    "Malta", "Marshall Islands", "Mauritania", "Mauritius", "Mexico",
    "Micronesia", "Moldova", "Monaco", "Mongolia", "Montenegro", "Morocco",
    "Mozambique", "Myanmar", "Namibia", "Nauru", "Nepal", "Netherlands",
    "New Zealand", "Nicaragua", "Niger", "Nigeria", "North Macedonia", "Norway",
    "Oman", "Pakistan", "Palau", "Panama", "Papua New Guinea", "Paraguay",
    "Peru", "Philippines", "Poland", "Portugal", "Qatar", "South Korea",
    "Romania", "Russia", "Rwanda", "Saint Kitts and Nevis", "Saint Lucia",
    "Saint Vincent and the Grenadines", "Samoa", "San Marino",
    "Sao Tome and Principe", "Saudi Arabia", "Senegal", "Serbia", "Seychelles",
    "Sierra Leone", "Singapore", "Slovakia", "Slovenia", "Solomon Islands",
    "Somalia", "South Africa", "South Sudan", "Spain", "Sri Lanka", "Sudan",
    "Suriname", "Sweden", "Switzerland", "Syria", "Tajikistan", "Tanzania",
    "Thailand", "Timor-Leste", "Togo", "Tonga", "Trinidad and Tobago", "Tunisia",
    "Türkiye", "Turkmenistan", "Tuvalu", "Uganda", "Ukraine",
    "United Arab Emirates", "United Kingdom", "United States of America", "Uruguay",
    "Uzbekistan", "Vanuatu", "Venezuela", "Vietnam", "Yemen", "Zambia", "Zimbabwe",
)

COUNTRY_ALIASES = {
    "Brunei Darussalam": "Brunei",
    "Congo": "Republic of the Congo",
    "Czech Republic": "Czechia",
    "Federated States of Micronesia": "Micronesia",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Principality of Liechtenstein": "Liechtenstein",
    "Republic of Moldova": "Moldova",
    "Russian Federation": "Russia",
    "Syrian Arab Republic": "Syria",
    "Turkey": "Türkiye",
    "United States": "United States of America",
    "Viet Nam": "Vietnam",
}


def normalize_country(value: object) -> str:
    """Strip whitespace and convert known aliases to one standard name."""
    name = str(value).strip()
    return COUNTRY_ALIASES.get(name, name)


# ------------------------------------------------------------------
# 3. Read and validate the raw data
# ------------------------------------------------------------------

support_path = find_input(SUPPORT_FILENAME)
cooperation_path = find_input(COOPERATION_FILENAME)

support = pd.read_excel(support_path, sheet_name="Explicit Support")
cooperation = pd.read_csv(cooperation_path, encoding="utf-8")

required_support_columns = {"Country Name", "Source"}
required_cooperation_columns = {
    "Country",
    "Area of Cooperation",
    "Area_Normalized",
}

if not required_support_columns.issubset(support.columns):
    raise ValueError(f"Support file must contain {sorted(required_support_columns)}")

if not required_cooperation_columns.issubset(cooperation.columns):
    raise ValueError(
        f"Cooperation file must contain {sorted(required_cooperation_columns)}"
    )

if support[["Country Name", "Source"]].isna().any().any():
    raise ValueError("The support file contains a missing country name or source.")

if cooperation[list(required_cooperation_columns)].isna().any().any():
    raise ValueError("The cooperation file contains missing analytical values.")

support = support.copy()
cooperation = cooperation.copy()
support["Country_Analysis"] = support["Country Name"].map(normalize_country)
cooperation["Country_Analysis"] = cooperation["Country"].map(normalize_country)
cooperation["Area of Cooperation"] = cooperation["Area of Cooperation"].astype(str).str.strip()
cooperation["Area_Normalized"] = cooperation["Area_Normalized"].astype(str).str.strip()

if support["Country_Analysis"].duplicated().any():
    duplicates = sorted(
        support.loc[
            support["Country_Analysis"].duplicated(keep=False),
            "Country_Analysis",
        ].unique()
    )
    raise ValueError(f"Duplicate support countries found: {duplicates}")

if cooperation.duplicated().any():
    raise ValueError("The cooperation file contains exactly duplicated rows.")


# ------------------------------------------------------------------
# 4. Construct the complete 192-country dataset
# ------------------------------------------------------------------

if len(UN_MEMBER_STATES) != 193 or len(set(UN_MEMBER_STATES)) != 193:
    raise AssertionError("The UN-member universe must contain 193 unique states.")

universe = sorted(set(UN_MEMBER_STATES) - {"India"})
universe_set = set(universe)

support_outside_universe = sorted(
    set(support["Country_Analysis"]) - universe_set
)
if support_outside_universe:
    raise ValueError(
        "Support countries not matched to the UN-member universe: "
        + ", ".join(support_outside_universe)
    )

# Keep UN members and transparently separate territories/unknown labels.
valid_cooperation = cooperation.loc[
    cooperation["Country_Analysis"].isin(universe_set)
].copy()

excluded_cooperation = cooperation.loc[
    ~cooperation["Country_Analysis"].isin(universe_set)
].copy()

cooperation_summary = (
    valid_cooperation
    .groupby("Country_Analysis", as_index=False)
    .agg(
        Cooperation_Entries=("Area of Cooperation", "size"),
        Cooperation_Breadth=("Area_Normalized", "nunique"),
    )
)

support_lookup = support[
    ["Country_Analysis", "Source"]
].rename(columns={"Source": "Support_Source"})

analysis = pd.DataFrame({"Country_Analysis": universe})
analysis = analysis.merge(
    cooperation_summary,
    on="Country_Analysis",
    how="left",
    validate="one_to_one",
)
analysis = analysis.merge(
    support_lookup,
    on="Country_Analysis",
    how="left",
    validate="one_to_one",
)

# Absence from the cooperation file means zero recorded cooperation—not a
# missing observation. This is the correction that produces N=192.
analysis[["Cooperation_Entries", "Cooperation_Breadth"]] = (
    analysis[["Cooperation_Entries", "Cooperation_Breadth"]]
    .fillna(0)
    .astype(int)
)

analysis["Documented_Support"] = analysis["Support_Source"].notna().astype(int)
analysis["Support_Source"] = analysis["Support_Source"].fillna("")
analysis["Support_Status"] = np.where(
    analysis["Documented_Support"].eq(1),
    "Documented explicit support",
    "No documented explicit support in supplied file",
)

if len(analysis) != 192:
    raise AssertionError(f"Expected 192 countries, found {len(analysis)}")

if analysis["Documented_Support"].sum() != support["Country_Analysis"].nunique():
    raise AssertionError("Not every documented supporter was retained exactly once.")

if analysis["Cooperation_Entries"].sum() != len(valid_cooperation):
    raise AssertionError("The aggregated cooperation counts do not match the raw rows.")


# ------------------------------------------------------------------
# 5. Descriptive statistics and Spearman correlations
# ------------------------------------------------------------------

descriptive_statistics = (
    analysis
    .groupby("Support_Status", as_index=False)
    .agg(
        N=("Country_Analysis", "size"),
        Mean_Entries=("Cooperation_Entries", "mean"),
        Median_Entries=("Cooperation_Entries", "median"),
        Mean_Breadth=("Cooperation_Breadth", "mean"),
        Median_Breadth=("Cooperation_Breadth", "median"),
    )
)

results = []
for outcome in ("Cooperation_Entries", "Cooperation_Breadth"):
    test = spearmanr(
        analysis["Documented_Support"],
        analysis[outcome],
        alternative="two-sided",
        nan_policy="raise",
    )
    results.append(
        {
            "Predictor": "Documented_Support",
            "Outcome": outcome,
            "N": len(analysis),
            "Documented_Support_N": int(analysis["Documented_Support"].sum()),
            "Spearman_rho": float(test.statistic),
            "p_value_two_sided": float(test.pvalue),
            "Significant_at_05": bool(test.pvalue < 0.05),
        }
    )

spearman_results = pd.DataFrame(results)

print("\nCorrected sample:")
print(f"Countries: {len(analysis)}")
print(f"Documented supporters: {int(analysis['Documented_Support'].sum())}")
print(f"No documented support: {int((analysis['Documented_Support'] == 0).sum())}")
print(f"Countries with active cooperation: {int((analysis['Cooperation_Entries'] > 0).sum())}")
print(f"Valid cooperation rows retained: {len(valid_cooperation)}")
print(f"Non-UN/unknown cooperation rows excluded: {len(excluded_cooperation)}")

print("\nDescriptive statistics:")
print(descriptive_statistics.round(3).to_string(index=False))

print("\nSpearman correlation results:")
print(
    spearman_results[
        ["Outcome", "N", "Spearman_rho", "p_value_two_sided", "Significant_at_05"]
    ].to_string(index=False, float_format=lambda value: f"{value:.6f}")
)


# ------------------------------------------------------------------
# 6. Save reproducible outputs
# ------------------------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
analysis.to_csv(OUTPUT_DIR / "country_level_unsc_support.csv", index=False)
descriptive_statistics.to_csv(
    OUTPUT_DIR / "descriptive_statistics_unsc_support.csv",
    index=False,
)
spearman_results.to_csv(
    OUTPUT_DIR / "spearman_unsc_support_results.csv",
    index=False,
)
excluded_cooperation.to_csv(
    OUTPUT_DIR / "excluded_non_un_cooperation_rows.csv",
    index=False,
)

print(f"\nFiles saved in: {OUTPUT_DIR}")

## Visualization

The two panels show the country-level distributions behind the Spearman correlations. Points are countries, boxes show the median and interquartile range, and the violin shapes summarize each distribution. Zero means no cooperation was recorded in the supplied data.

In [ ]:
# ------------------------------------------------------------------
# 7. Publication-ready visualization
# ------------------------------------------------------------------

import matplotlib.pyplot as plt

plot_specs = [
    {
        "column": "Cooperation_Entries",
        "title": "Cooperation entries",
        "ylabel": "Number of cooperation entries",
    },
    {
        "column": "Cooperation_Breadth",
        "title": "Cooperation breadth",
        "ylabel": "Number of distinct cooperation areas",
    },
]

group_order = [0, 1]
group_labels = {
    0: "No documented\nsupport",
    1: "Documented\nsupport",
}
group_colors = {0: "#778795", 1: "#C75C3A"}
rng = np.random.default_rng(42)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titleweight": "bold",
    "axes.edgecolor": "#30343B",
    "axes.labelcolor": "#30343B",
    "xtick.color": "#30343B",
    "ytick.color": "#30343B",
})

fig, axes = plt.subplots(1, 2, figsize=(11.2, 5.9), constrained_layout=False)
fig.patch.set_facecolor("white")

for ax, spec in zip(axes, plot_specs):
    column = spec["column"]
    grouped_values = [
        analysis.loc[analysis["Documented_Support"].eq(group), column].to_numpy()
        for group in group_order
    ]

    # Violin shapes summarize the full distributions.
    violins = ax.violinplot(
        grouped_values,
        positions=[0, 1],
        widths=0.72,
        showmeans=False,
        showmedians=False,
        showextrema=False,
        bw_method=0.45,
    )
    for group, body in zip(group_order, violins["bodies"]):
        body.set_facecolor(group_colors[group])
        body.set_edgecolor(group_colors[group])
        body.set_alpha(0.20)
        body.set_linewidth(1.0)

    # Boxplots make the median and interquartile range easy to compare.
    boxes = ax.boxplot(
        grouped_values,
        positions=[0, 1],
        widths=0.20,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "white", "linewidth": 2.2},
        whiskerprops={"color": "#4B515B", "linewidth": 1.2},
        capprops={"color": "#4B515B", "linewidth": 1.2},
    )
    for group, box in zip(group_order, boxes["boxes"]):
        box.set_facecolor(group_colors[group])
        box.set_edgecolor("#4B515B")
        box.set_linewidth(1.0)

    # Jittered points keep every country visible, including zeros.
    for position, (group, values) in enumerate(zip(group_order, grouped_values)):
        jitter = rng.normal(loc=position, scale=0.055, size=len(values))
        ax.scatter(
            jitter,
            values,
            s=25,
            color=group_colors[group],
            alpha=0.55,
            edgecolor="white",
            linewidth=0.45,
            zorder=3,
        )

    result = spearman_results.loc[
        spearman_results["Outcome"].eq(column)
    ].iloc[0]
    rho = result["Spearman_rho"]
    p_value = result["p_value_two_sided"]
    p_text = "p < 0.001" if p_value < 0.001 else f"p = {p_value:.3f}"

    ax.text(
        0.04,
        0.95,
        f"Spearman $\\rho$ = {rho:.3f}\n{p_text}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10.5,
        color="#30343B",
        bbox={
            "boxstyle": "round,pad=0.45",
            "facecolor": "white",
            "edgecolor": "#D8DDE3",
            "alpha": 0.96,
        },
    )

    counts = [len(values) for values in grouped_values]
    ax.set_xticks([0, 1])
    ax.set_xticklabels([
        f"{group_labels[group]}\n(n = {count})"
        for group, count in zip(group_order, counts)
    ])
    ax.set_title(spec["title"], fontsize=13, loc="left", pad=13)
    ax.set_ylabel(spec["ylabel"])
    ax.set_xlim(-0.55, 1.55)
    ax.set_ylim(bottom=-0.55)
    ax.grid(axis="y", color="#E7EAEE", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "Documented UNSC support is associated with greater science cooperation",
    x=0.07,
    y=0.975,
    ha="left",
    fontsize=16,
    fontweight="bold",
    color="#22262D",
)
fig.text(
    0.07,
    0.91,
    "Complete sample of 192 non-India UN members; two-sided Spearman tests",
    ha="left",
    fontsize=10.5,
    color="#606873",
)
fig.text(
    0.07,
    0.025,
    "Points represent countries; boxes show median and IQR. Zero denotes no recorded cooperation in the supplied data.",
    ha="left",
    fontsize=9,
    color="#606873",
)
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.18, top=0.82, wspace=0.30)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
png_path = OUTPUT_DIR / "unsc_support_science_cooperation.png"
svg_path = OUTPUT_DIR / "unsc_support_science_cooperation.svg"
fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(svg_path, bbox_inches="tight", facecolor="white")
plt.show()

print(f"High-resolution PNG saved to: {png_path}")
print(f"Editable SVG saved to: {svg_path}")